# 00 · Preparar o ambiente e modelar a base em SQL
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/previsao-faltas-consultas/blob/main/notebooks/00_setup_modelagem_sql.ipynb)

**O que este notebook faz:**
1. Conecta ao Google Drive.
2. Baixa o CSV original (que está neste repositório) e guarda uma cópia na pasta `dados` do projeto.
3. **Modela a base em 3 tabelas** e grava em um banco SQLite (`consultas.db`):
   - `pacientes`: uma linha por paciente (gênero, bolsa família, condições de saúde)
   - `bairros`: uma linha por bairro
   - `consultas`: uma linha por agendamento (datas, idade, SMS, faltou)

A base original vem em uma tabela só. Separar em tabelas permite praticar JOINs, como num banco de verdade.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sqlite3, pandas as pd
BASE  = '/content/drive/MyDrive/03_Portfolio/Projetos_ML/02_previsao-faltas-consultas'
DADOS = f'{BASE}/dados'
os.makedirs(DADOS, exist_ok=True)

URL = 'https://raw.githubusercontent.com/jessicapmartins/previsao-faltas-consultas/main/data/raw/KaggleV2-May-2016.csv'
bruto = pd.read_csv(URL)
bruto.to_csv(f'{DADOS}/KaggleV2-May-2016.csv', index=False)
print(bruto.shape)
bruto.head()

In [ ]:
df = bruto.rename(columns={
    'PatientId':'paciente_id','AppointmentID':'consulta_id','Gender':'genero',
    'ScheduledDay':'data_agendamento','AppointmentDay':'data_consulta','Age':'idade',
    'Neighbourhood':'bairro','Scholarship':'bolsa_familia','Hipertension':'hipertensao',
    'Diabetes':'diabetes','Alcoholism':'alcoolismo','Handcap':'deficiencia',
    'SMS_received':'sms_recebido','No-show':'faltou'})
df['paciente_id'] = df['paciente_id'].astype('int64')
df['faltou'] = (df['faltou'] == 'Yes').astype(int)
df['data_agendamento'] = pd.to_datetime(df['data_agendamento']).dt.tz_localize(None)
df['data_consulta']    = pd.to_datetime(df['data_consulta']).dt.tz_localize(None).dt.date

bairros = (df[['bairro']].drop_duplicates().sort_values('bairro')
             .reset_index(drop=True).rename_axis('bairro_id').reset_index())
pacientes = df[['paciente_id','genero','bolsa_familia','hipertensao','diabetes','alcoolismo','deficiencia']].drop_duplicates('paciente_id')
consultas = (df.merge(bairros, on='bairro')
               [['consulta_id','paciente_id','bairro_id','data_agendamento','data_consulta','idade','sms_recebido','faltou']])

con = sqlite3.connect(f'{DADOS}/consultas.db')
for nome, tab in {'bairros':bairros, 'pacientes':pacientes, 'consultas':consultas}.items():
    tab.to_sql(nome, con, if_exists='replace', index=False)
    print(f'{nome:<10} {len(tab):>8,} linhas')
con.execute('CREATE INDEX IF NOT EXISTS ix_cons_pac ON consultas(paciente_id)')
con.commit()

### Conferência
Esperado: 81 bairros, 62.299 pacientes e 110.527 consultas, com cerca de 20% de faltas.

In [ ]:
pd.read_sql('SELECT AVG(faltou) AS taxa_falta FROM consultas', con)